# 8 · Dataloaders

A model never reads the `Dataset` directly while it trains: it asks one of the
dataset's entities for a PyTorch `DataLoader`, and the evaluator does the same
for the evaluation split. This guide opens every loader WarpRec builds and looks
at what one batch holds. It covers:

- the configuration keys that reach the loaders, run through
  `load_design_configuration` and `initialize_datasets`;
- the four loaders over the interaction matrix (dense user rows, pointwise,
  contrastive triplets, positive pairs) and which models read each;
- the row loader that keeps contexts attached to their interaction;
- the five session loaders the sequential models train on, and the history
  lookup they are scored with;
- negative sampling: `uniform` against `popularity` and `neg_alpha`, measured;
- seeding: what a seed fixes and what changes between epochs;
- the four evaluation loaders: full, sampled, and their contextual variants.

**Requirements:** `pip install warprec jupyter`. Runs in under a minute on
MovieLens-100K (`guide_data.movielens_100k()` downloads and converts it exactly
as guide 1 shows).

In [1]:
import hashlib
import inspect
import re
import sys
import textwrap
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()
titles = pd.read_csv(folder / "items.tsv", sep="\t").set_index("item_id")["title"]
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

## The keys that reach the loaders

Three parts of a configuration decide what the loaders yield:

- `training` holds how negatives are drawn, for every model that samples them;
- each model's own hyperparameters hold its loader's `batch_size`, and for some
  models `neg_samples`, `max_seq_len` or `mask_prob`;
- `evaluation` holds the evaluation loader's `batch_size` and, under
  `strategy: sampled`, how many candidates are drawn and how.

In [2]:
print(Path("configs/loaders.yml").read_text())

# The keys that decide what the training and evaluation loaders yield.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
# How training negatives are drawn, for every model that samples them.
training:
  negative_sampling: popularity
  neg_alpha: 0.75
# The batch size of a training loader is a hyperparameter of each model.
models:
  BPR:
    embedding_size: 64
    reg_weight: 0.001
    batch_size: 2048
    epochs: 1
    learning_rate: 0.001
  SASRec:
    embedding_size: 64
    n_layers: 2
    n_heads: 2
    inner_size: 128
    dropout_prob: 0.2
    attn_dropout_prob: 0.2
    reg_weight: 0.0
    weight_decay: 0.0
    batch_size: 512
    epochs: 1
    learning_rate: 0.001
    neg_samples: 1
    max_seq_len: 50
# The evaluation loaders: their batch size and, for 'sampled', the candidates.
evaluation:
  top_k: [10]
  

`initialize_datasets` reads and splits the data and builds the `Dataset`. The
`training` options are stored on the training entities, and
`evaluation.batch_size` becomes the dataset's own `batch_size`, the one its
evaluation loaders use.

In [3]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration

logger.disable("warprec")
config = load_design_configuration("configs/loaders.yml")
main, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
n_users, n_items = main.get_dims()
print(
    f"{n_users} users, {n_items} items, {main.train_set.get_sparse().nnz} training interactions"
)
print(
    "negative_sampling:",
    main.train_set.negative_sampling,
    "| neg_alpha:",
    main.train_set.neg_alpha,
    "| evaluation batch_size:",
    main.batch_size,
)

943 users, 1648 items, 89561 training interactions
negative_sampling: popularity | neg_alpha: 0.75 | evaluation batch_size: 256


The loaders live on three entities of the dataset, built once by the
constructor:

| attribute | entity | what it holds | loaders |
|---|---|---|---|
| `train_set` | `Interactions` | the sparse user × item matrix | interaction, pointwise, contrastive, positive |
| `train_transactions` | `Transactions` | one row per record, with its contexts; only when contexts are configured | pointwise |
| `train_session` | `Sessions` | each user's items in timestamp order | sequential, same-target, sliding window, cloze |

The pipelines never pick a loader by name. They build the model and call its
`get_dataloader(interactions, sessions)`, which knows which loader the model
needs and passes its own hyperparameters. This is what the design pipeline does
for each model of the configuration:

In [4]:
from warprec.utils.registry import model_registry

bpr = model_registry.get(
    "BPR",
    params=config.models["BPR"],
    interactions=main.train_set,
    sessions=main.train_session,
    info=main.info(),
)
loader = bpr.get_dataloader(main.train_set, main.train_session)
print(
    type(loader.dataset).__name__,
    "| batch_size",
    loader.batch_size,
    "| batches",
    len(loader),
)

ContrastiveDataset | batch_size 2048 | batches 44


BPR asks `train_set` for its contrastive loader, with the `batch_size: 2048` of
the configuration. The rest of this guide calls the entities directly, with
small batches and `shuffle=False` so the output can be read. Every loader takes
`batch_size`, `shuffle`, `seed` and any other `DataLoader` argument
(`num_workers`, `pin_memory`, …).

A small helper prints a batch: one line per tensor, with its shape, dtype and
first row.

In [5]:
user_ids, item_ids = main.get_inverse_mappings()


def show(batch, names):
    """Shape, dtype and the first row of every tensor of a batch."""

    def first(t):
        values = t[0].flatten().tolist() if t.dim() else [t.item()]
        values = [round(v, 3) if isinstance(v, float) else v for v in values]
        return str(values[:8])[:-1] + (", …]" if len(values) > 8 else "]")

    return pd.DataFrame(
        {
            "shape": [tuple(t.shape) for t in batch],
            "dtype": [str(t.dtype).removeprefix("torch.") for t in batch],
            "first row": [first(t) for t in batch],
        },
        index=names,
    )


def raw(items):
    """Item indices back to MovieLens ids, dropping the padding index."""
    return [item_ids[int(i)] for i in items if int(i) < n_items]

## Interactions: the matrix loaders

### Dense user rows

`get_interaction_dataloader` yields whole rows of the training matrix, densified
one at a time from the sparse matrix: one sample per user, `n_items` wide. With
`include_user_id=True` the user index comes first. The autoencoders read it.

In [6]:
batch = next(
    iter(
        main.train_set.get_interaction_dataloader(
            include_user_id=True, batch_size=4, shuffle=False
        )
    )
)
show(batch, ["user", "row"])

,shape,dtype,first row
user,"(4,)",int64,[0]
row,"(4, 1648)",float32,"[1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, …]"


In [7]:
user, row = batch[0][0].item(), batch[1][0]
print(
    f"index {user} is user {user_ids[user]}: {int(row.sum())} training items,",
    f"first ones {raw(row.nonzero().flatten()[:5])}",
)

index 0 is user 1: 244 training items, first ones [1, 2, 3, 4, 7]


### Pointwise rows

`get_pointwise_dataloader(neg_samples=n)` yields `(user, item, rating)` rows.
Each training interaction becomes a block of `1 + n` consecutive samples: the
positive with label 1, then `n` negatives of the same user with label 0. The
label is 1 for a positive whatever `reader.rating_type` is, so the loader is
for implicit objectives. With `shuffle=False` the blocks are visible:

In [8]:
pointwise = main.train_set.get_pointwise_dataloader(
    neg_samples=2, batch_size=6, shuffle=False
)
batch = next(iter(pointwise))
print("samples:", len(pointwise.dataset), "= interactions x 3")
pd.DataFrame([t.tolist() for t in batch], index=["user", "item", "label"], dtype=object)

samples: 268683 = interactions x 3


,0,1,2,3,4,5
user,0,0,0,0,0,0
item,0,742,318,1,927,637
label,1.0,0.0,0.0,1.0,0.0,0.0


`include_side_info=True` appends the item's encoded side information to every
row when the dataset has any (guide 4).

### Contrastive triplets

`get_contrastive_dataloader` yields one `(user, positive, negative)` triplet
per training interaction, the negative drawn on the fly. This is what BPR and
most graph, knowledge-aware and multimodal models read.

In [9]:
contrastive = main.train_set.get_contrastive_dataloader(batch_size=4, shuffle=False)
batch = next(iter(contrastive))
show(batch, ["user", "positive", "negative"])

,shape,dtype,first row
user,"(4,)",int64,[0]
positive,"(4,)",int64,[0]
negative,"(4,)",int64,[742]


In [10]:
u, p, n = (t[0].item() for t in batch)
print(f"user {user_ids[u]}: watched {item_ids[p]} {titles[item_ids[p]]!r},")
print(f"  negative {item_ids[n]} {titles[item_ids[n]]!r}")

user 1: watched 1 'Toy Story (1995)',
  negative 743 'Crow: City of Angels, The (1996)'


### Positive pairs

`get_positive_dataloader` yields `(user, positive)` pairs and draws nothing.
The models that read it (TwoTower, BM3, RecDCL) build their losses over the
batch rather than from sampled negatives; TwoTower, for instance, treats the
other items of the batch as negatives.

In [11]:
positive = main.train_set.get_positive_dataloader(batch_size=4, shuffle=False)
show(next(iter(positive)), ["user", "positive"])

,shape,dtype,first row
user,"(4,)",int64,[0]
positive,"(4,)",int64,[0]


### Which model reads which loader

The table below is read from the code: for every registered model it inspects
the `get_dataloader` the model inherits or defines and records the loader it
calls. Models that are not iterative (EASE, ItemKNN, iALS, …) fit in closed form
and have no training loader.

In [12]:
from warprec.recommenders.base_recommender import IterativeRecommender

by_loader = defaultdict(list)
for name in sorted(model_registry.list_registered(), key=str.lower):
    cls = model_registry.get_class(name)
    if not issubclass(cls, IterativeRecommender):
        by_loader["(no loader: closed form)"].append(cls.__name__)
        continue
    source = inspect.getsource(cls.get_dataloader)
    called = re.findall(r"\.(get_\w+_dataloader)\(", source)[0]
    if "include_context" in source:
        called += " (Transactions)"
    by_loader[called].append(cls.__name__)

for called, models in sorted(by_loader.items(), key=lambda kv: -len(kv[1])):
    print(f"{called} — {len(models)} models")
    print(
        textwrap.fill(
            ", ".join(models), 88, initial_indent="    ", subsequent_indent="    "
        )
    )

get_contrastive_dataloader — 30 models
    BPR, CKE, ConvNCF, DGCF, EGCF, ESIGCF, FREEDOM, KaHFM, KGAT, KGCN, KGFlex, KGIN,
    LATTICE, LightCCF, LightGCL, LightGCN, LightGCNpp, MGCN, MixRec, MMGCN, NGCF, PAAC,
    PopDCL, RippleNet, SGL, SimGCL, SimRec, UltraGCN, VBPR, XSimGCL
(no loader: closed form) — 20 models
    AddEASE, ADMMSlim, AttributeItemKNN, AttributeUserKNN, CEASE, EASE, iALS, iALS2008,
    ItemKNN, ItemKNNTD, Pop, ProxyRecommender, Random, RP3Beta, SANSA, Slim, STAN,
    UserKNN, UserKNNTD, VSM
get_sequential_dataloader — 11 models
    BSARec, Caser, CL4SRec, CORE, eSASRec, FOSSIL, GRU4Rec, LightSANs, LinRec, NARM,
    SASRec
get_pointwise_dataloader (Transactions) — 8 models
    AFM, DCN, DCNv2, DeepFM, FM, NFM, WideAndDeep, xDeepFM
get_pointwise_dataloader — 7 models
    FISM, GCMC, LightGODE, MACRGCN, MACRMF, NeuMF, SGCL
get_interaction_dataloader — 6 models
    CDAE, DiffRec, ELSA, MacridVAE, MultiDAE, MultiVAE
get_positive_dataloader — 3 models
    BM3, RecDCL, Two

## Transactions: rows with their contexts

A matrix holds one cell per (user, item) pair, so it cannot carry the same pair
seen in two contexts. When contexts are configured, the dataset also builds
`train_transactions`, which keeps every row, and the context-aware models
(FM, DeepFM, …) read its pointwise loader with `include_context=True`. Guide 5
covers contexts; here two are derived from the timestamp on the same split and
passed to the `Dataset` constructor (in a configuration they are
`reader.labels.context_labels`).

In [13]:
from warprec.data import Dataset


def with_context(frame):
    frame = frame.to_pandas()
    when = pd.to_datetime(frame["timestamp"], unit="s")
    return frame.assign(
        weekend=(when.dt.dayofweek >= 5).astype(int), daypart=when.dt.hour // 6
    )


train_frame, test_frame = main.train_set.get_df(), main.eval_set.get_df()
contextual = Dataset(
    train_data=with_context(train_frame),
    eval_data=with_context(test_frame),
    timestamp_label="timestamp",
    context_labels=["weekend", "daypart"],
)
rows = contextual.train_transactions.get_pointwise_dataloader(
    neg_samples=1, include_context=True, batch_size=4, shuffle=False
)
show(next(iter(rows)), ["user", "item", "label", "context"])

,shape,dtype,first row
user,"(4,)",int64,[0]
item,"(4,)",int64,[0]
label,"(4,)",float32,[1.0]
context,"(4, 2)",float32,"[1.0, 4.0]"


The context column holds each field's encoded value (index 0 is reserved for
unknown values, so `weekend` is 1 or 2 and `daypart` 1 to 4). A negative carries
the context of the positive it was drawn for: it is "an item the user did not
pick in that situation".

`Interactions` refuses the same request rather than attach contexts to the wrong
rows:

In [14]:
try:
    contextual.train_set.get_pointwise_dataloader(include_context=True)
except NotImplementedError as error:
    print(str(error)[:111], "…")

Interactions cannot supply contexts: its rows come from the interaction matrix, which holds one cell per (user, …


## Sessions: the sequential loaders

`train_session` orders each user's training items by `timestamp` and keeps them
on one flat array. Every session loader pads with index `n_items`, which is
never an item.

### Next item

`get_sequential_dataloader` yields one sample per training interaction except
each user's first: the items before it (the most recent `max_seq_len` of them,
left-aligned and padded on the right), the length, and the item itself as the
target. `neg_samples` adds that many negatives per sample; `include_user_id`
puts the user index first, for Caser and FOSSIL.

In [15]:
sequential = main.train_session.get_sequential_dataloader(
    max_seq_len=5, neg_samples=2, include_user_id=True, batch_size=6, shuffle=False
)
batch = next(iter(sequential))
print(
    "samples:",
    len(sequential.dataset),
    "=",
    main.train_set.get_sparse().nnz,
    "-",
    n_users,
    "first items",
)
show(batch, ["user", "sequence", "length", "target", "negatives"])

samples: 88618 = 89561 - 943 first items


,shape,dtype,first row
user,"(6,)",int64,[0]
sequence,"(6, 5)",int64,"[167, 1648, 1648, 1648, 1648]"
length,"(6,)",int64,[1]
target,"(6,)",int64,[171]
negatives,"(6, 2)",int64,"[1275, 1078]"


The first six samples all belong to the first user: the history grows by one
item per sample until it reaches `max_seq_len`, then slides. Decoded, the sixth
sample has the user's seventh film as its target and the five before it as its
history; the first film has slid out:

In [16]:
print("history:", raw(batch[1][5]), "-> target:", raw(batch[3][5:6]))
ratings = pd.read_csv(folder / "ratings.tsv", sep="\t")
first_user = ratings[ratings.user_id == user_ids[batch[0][0].item()]]
first_user.sort_values("timestamp", kind="stable").head(7)

history: [172, 165, 156, 196, 166] -> target: [187]


,user_id,item_id,rating,timestamp
59972,1,168,5,874965478
92487,1,172,5,874965478
74577,1,165,5,874965518
48214,1,156,4,874965556
15764,1,196,5,874965677
22971,1,166,5,874965677
21626,1,187,4,874965678


The two agree, and the table shows a property of MovieLens worth knowing before
training a sequential model on it: timestamps are in seconds, yet ratings often
share theirs with another rating of the same user (here the first two, and the
fifth and sixth). `Sessions` sorts by `(user, timestamp)` only, so the order
inside such a tie is the sort's, not the data's. Across the dataset:

In [17]:
ties = ratings.groupby(["user_id", "timestamp"]).size()
print(
    f"{ties[ties > 1].sum():,} of {len(ratings):,} ratings share their timestamp with another of the same user"
)

75,772 of 100,000 ratings share their timestamp with another of the same user


### Same target

`get_same_target_sequential_dataloader` (DuoRec) adds, to each next-item
sample, the history of another sample that ends in the same target, drawn at
random, and a flag that is `False` when the target occurs only once and the
sample had to be paired with itself.

In [18]:
same_target = main.train_session.get_same_target_sequential_dataloader(
    max_seq_len=5, batch_size=6, shuffle=False
)
show(
    next(iter(same_target)),
    ["sequence", "length", "target", "same-target sequence", "its length", "found"],
)

,shape,dtype,first row
sequence,"(6, 5)",int64,"[167, 1648, 1648, 1648, 1648]"
length,"(6,)",int64,[1]
target,"(6,)",int64,[171]
same-target sequence,"(6, 5)",int64,"[287, 68, 482, 190, 1648]"
its length,"(6,)",int64,[4]
found,"(6,)",bool,[True]


### Sliding windows

`get_sliding_window_dataloader` (gSASRec) yields windows of `max_seq_len`
consecutive items, moved by `stride` along each user's history, with
`neg_samples` negatives for every position. The model predicts each position
from the ones before it, so a window is input and target at once. A user with
`L >= 2` items gives `max(L - max_seq_len, 0) // stride + 1` windows.

In [19]:
windows = main.train_session.get_sliding_window_dataloader(
    max_seq_len=5, neg_samples=2, stride=5, batch_size=4, shuffle=False
)
batch = next(iter(windows))
print("windows:", len(windows.dataset))
show(batch, ["window", "negatives"])

windows: 17524


,shape,dtype,first row
window,"(4, 5)",int64,"[167, 171, 164, 155, 195]"
negatives,"(4, 5, 2)",int64,"[1275, 1078, 723, 713, 1414, 1149, 332, 867, …]"


### Cloze masking

`get_cloze_mask_dataloader` (BERT4Rec) yields one sample per user: the last
`max_seq_len` items, with a `mask_prob` share of the positions (at least one)
replaced by `mask_token_id`. BERT4Rec uses `n_items + 1`, the index after the
padding. Alongside come the original items at the masked positions, their
negatives and the positions themselves, each padded to `max_seq_len`.

In [20]:
cloze = main.train_session.get_cloze_mask_dataloader(
    max_seq_len=10,
    mask_prob=0.2,
    mask_token_id=n_items + 1,
    neg_samples=1,
    batch_size=4,
    shuffle=False,
)
batch = next(iter(cloze))
print(
    "samples:", len(cloze.dataset), "| mask token:", n_items + 1, "| padding:", n_items
)
show(batch, ["masked sequence", "targets", "negatives", "positions"])

samples: 943 | mask token: 1649 | padding: 1648


,shape,dtype,first row
masked sequence,"(4, 10)",int64,"[1649, 74, 141, 50, 53, 227, 43, 1649, …]"
targets,"(4, 10)",int64,"[138, 85, 1648, 1648, 1648, 1648, 1648, 1648, …]"
negatives,"(4, 10, 1)",int64,"[723, 713, 1648, 1648, 1648, 1648, 1648, 1648, …]"
positions,"(4, 10)",int64,"[0, 7, 0, 0, 0, 0, 0, 0, …]"


`positions` pads with 0, which is also a real position; the number of targets of
a sample is where `targets` differs from the padding index.

### The history a sequential model is scored with

At evaluation (and when serving) a sequential model is given each user's most
recent training items, through `get_user_history_sequences(users,
max_seq_len)`. It returns the sequences, right-padded to the longest in the
batch, and their lengths. Unlike the training samples, the last training item is
part of the input: the target is the test split.

In [21]:
sequences, lengths = main.train_session.get_user_history_sequences(
    [0, 1, 2], max_seq_len=5
)
print(sequences, lengths, sep="\n")
print("user", user_ids[0], "most recent:", raw(sequences[0]))

tensor([[227,  43,  85,  99, 153],
        [284, 301, 309, 271, 299],
        [326, 320, 259, 328, 346]])
tensor([5, 5, 5])
user 1 most recent: [228, 44, 86, 100, 154]


## Negative sampling

The interaction and transaction samplers draw from `training.negative_sampling`:

- `uniform`: every item equally likely;
- `popularity`: item `i` with probability proportional to `count_i ** neg_alpha`,
  where `count_i` is its number of training interactions. `neg_alpha: 0` is
  uniform over the training catalogue, `1` is proportional to the counts.

Either way, a draw the user has already interacted with in training is
rejected and drawn again. Below, the first 20 batches of 1,024 triplets of the
contrastive loader are drawn under each setting, from datasets built on the same
split with the Python API. For every setting the table gives
the mean training count of the negatives, the share of them among the 10% most
popular items, and two checks: negatives the user has in training (must be 0)
and negatives that are in the user's *test* set. The sampler only knows the
training split, so the latter can happen.

In [22]:
train_matrix = main.train_set.get_sparse()
test_matrix = main.eval_set.get_sparse()
popularity = np.asarray(train_matrix.sum(axis=0)).ravel()
head = popularity >= np.quantile(popularity, 0.9)


def measure(users, negatives):
    users, negatives = users.numpy().copy(), negatives.numpy().copy()
    return {
        "mean count": round(float(popularity[negatives].mean()), 1),
        "share in top 10%": f"{head[negatives].mean():.1%}",
        "seen in training": int(train_matrix[users, negatives].sum()),
        "in test set": int(test_matrix[users, negatives].sum()),
    }


def draw(loader, batches=20):
    """The tensors of the first batches of a loader, concatenated."""
    return [torch.cat(t) for t in zip(*(b for _, b in zip(range(batches), loader)))]


settings = {"uniform": ("uniform", 0.75)} | {
    f"popularity, alpha={a}": ("popularity", a) for a in (0, 0.5, 0.75, 1)
}
table = {}
for label, (strategy, alpha) in settings.items():
    dataset = Dataset(
        train_data=train_frame,
        eval_data=test_frame,
        negative_sampling=strategy,
        neg_alpha=alpha,
    )
    users, _, negatives = draw(
        dataset.train_set.get_contrastive_dataloader(batch_size=1024)
    )
    table[label] = measure(users, negatives)

positive_items = train_matrix.indices
table["(all training positives)"] = {
    "mean count": round(float(popularity[positive_items].mean()), 1),
    "share in top 10%": f"{head[positive_items].mean():.1%}",
    "seen in training": "-",
    "in test set": "-",
}
pd.DataFrame(table).T

,mean count,share in top 10%,seen in training,in test set
uniform,43.7,6.4%,0,309
"popularity, alpha=0",43.2,6.3%,0,328
"popularity, alpha=0.5",85.4,17.7%,0,569
"popularity, alpha=0.75",107.0,25.0%,0,617
"popularity, alpha=1",127.1,32.6%,0,750
(all training positives),158.5,43.8%,-,-


Uniform and `alpha=0` give the same distribution, as they should (a mean count of
43.7 and 43.2). The mean then rises with `alpha`, to 127.1 at `alpha=1`. At that
value the draw follows exactly the distribution of the training positives
(158.5); the negatives stay below it because the popular items are the ones a
user has most often seen already, so they are rejected most often. No negative is
an item the user has in training, while the share that falls in the user's test
set grows with `alpha` (309 to 750 of 20,480).

The session loaders do not read `training.negative_sampling`: their negatives
are always drawn uniformly from the items the user has not seen in training
(and are never the target). The same measure on the sequential loader of the
dataset configured with `popularity`:

In [23]:
loader = main.train_session.get_sequential_dataloader(
    max_seq_len=5, neg_samples=1, include_user_id=True, batch_size=1024
)
users, _, _, _, negatives = draw(loader)
pd.DataFrame(
    {"sequential, popularity configured": measure(users, negatives.squeeze(1))}
).T

,mean count,share in top 10%,seen in training,in test set
"sequential, popularity configured",43.9,6.4%,0,314


## Seeding

Every loader takes a `seed`. It seeds two independent streams: the `torch`
generator that shuffles the samples, and the sampler's own NumPy generator that
draws the negatives. Neither touches the global random state, so other code
drawing random numbers in between does not change the batches. Below, the first
three batches of a contrastive loader are hashed under different seeds:

In [24]:
def fingerprint(loader, n=3):
    """A short hash of each of the first n batches."""
    hashes = []
    for _, batch in zip(range(n), loader):
        hashes.append(
            hashlib.sha1(b"".join(t.numpy().tobytes() for t in batch)).hexdigest()[:8]
        )
    return hashes


def contrastive_loader(seed):
    return main.train_set.get_contrastive_dataloader(batch_size=1024, seed=seed)


first = contrastive_loader(42)
epoch_1 = fingerprint(first)
np.random.seed(0), torch.manual_seed(0)  # disturbing the global streams changes nothing
pd.DataFrame(
    {
        "seed 42": epoch_1,
        "seed 42, new loader": fingerprint(contrastive_loader(42)),
        "seed 7": fingerprint(contrastive_loader(7)),
        "seed 42, second epoch": fingerprint(first),
    },
    index=["batch 1", "batch 2", "batch 3"],
).T

,batch 1,batch 2,batch 3
seed 42,c26b74ad,52c80c9f,c1ee860c
"seed 42, new loader",c26b74ad,52c80c9f,c1ee860c
seed 7,f92d958f,5f6d26db,db001874
"seed 42, second epoch",a7fe0f6c,ec173ea9,1e8b3d24


The same seed gives the same batches from a new loader; another seed gives
others. Iterating the same loader again is a new epoch: both streams continue,
so the order and the negatives change, while the whole run stays reproducible.
With `shuffle=False` only the order is fixed; the negatives still come from the
seed.

## Evaluation loaders

The evaluation loaders are built by the `Dataset`, over its evaluation split,
with the dataset's `batch_size` (`evaluation.batch_size`). Each is cached by its
arguments, so asking twice returns the same loader:

In [25]:
full = main.get_evaluation_dataloader()
print(
    full is main.get_evaluation_dataloader(),
    "| users:",
    len(full.dataset),
    "| batches:",
    len(full),
)

True | users: 943 | batches: 4


### Full

The full loader yields the test split of a batch of users as sparse
coordinates: the user indices, then for every test interaction its row within
the batch, its item and its value. The evaluator densifies it on the device and
scores the users against the whole catalogue. Users without test interactions
are left out.

In [26]:
batch = next(iter(full))
show(batch, ["users", "rows", "items", "values"])

,shape,dtype,first row
users,"(256,)",int64,[0]
rows,"(2650,)",int64,[0]
items,"(2650,)",int64,[4]
values,"(2650,)",float32,[1.0]


In [27]:
users, rows, items, values = batch
print(f"user {user_ids[users[0].item()]}: test items {raw(items[rows == 0])}")

user 1: test items [5, 6, 9, 16, 18, 20, 32, 74, 87, 102, 111, 129, 169, 171, 178, 189, 209, 221, 222, 242, 244, 255, 256, 258, 266, 270, 271, 272]


### Sampled

`get_sampled_evaluation_dataloader` draws, once and for every user, `num_negatives`
items the user has in neither the training nor the test split. A batch holds the
users, their test items padded to the longest list with `n_items`, and the
candidates; the model ranks each test item against them.

In [28]:
sampled = main.get_sampled_evaluation_dataloader(num_negatives=99, seed=42)
batch = next(iter(sampled))
show(batch, ["users", "test items", "candidates"])

,shape,dtype,first row
users,"(256,)",int64,[0]
test items,"(256, 63)",int64,"[4, 5, 8, 15, 17, 19, 31, 73, …]"
candidates,"(256, 99)",int64,"[1126, 1459, 860, 1294, 1130, 1095, 1044, 1638, …]"


In [29]:
users = torch.cat([b[0] for b in sampled])
candidates = torch.cat([b[2] for b in sampled])
pairs = (
    users.repeat_interleave(99).numpy().copy(),
    candidates.flatten().numpy().copy(),
)
print(
    "candidates seen in training or test:",
    int((train_matrix + test_matrix)[pairs].sum()),
)
seen = (train_matrix + test_matrix).tolil().rows
expected = np.mean(
    [np.setdiff1d(np.arange(n_items), seen[u]).mean() for u in users.tolist()]
)
print(f"mean candidate index: {candidates.float().mean():.1f}, expected {expected:.1f}")

candidates seen in training or test: 0


mean candidate index: 855.1, expected 852.7


The candidates never include an item the user has interacted with, and they are
drawn uniformly over each user's unseen items: their mean index (855.1) matches
the mean index of those items, averaged over the users (852.7).
`negative_sampling="popularity"` and `neg_alpha` switch the draw to the
popularity distribution, as in training; the mean training count of the
candidates goes from 47.0 to 111.4:

In [30]:
by_popularity = main.get_sampled_evaluation_dataloader(
    num_negatives=99, seed=42, negative_sampling="popularity", neg_alpha=0.75
)
pd.DataFrame(
    {
        name: {
            "mean count": round(
                float(popularity[torch.cat([b[2] for b in loader]).numpy()].mean()), 1
            )
        }
        for name, loader in [
            ("uniform", sampled),
            ("popularity, alpha=0.75", by_popularity),
        ]
    }
).T

,mean count
uniform,47.0
"popularity, alpha=0.75",111.4


In a pipeline the loader is picked by `retrieve_evaluation_dataloader` from
`evaluation.strategy` and the kind of model. With the configuration of this
guide it returns the sampled loader above, from the cache:

In [31]:
from warprec.utils.helpers import retrieve_evaluation_dataloader

chosen = retrieve_evaluation_dataloader(
    main,
    bpr,
    config.evaluation.strategy,
    num_negatives=config.evaluation.num_negatives,
    negative_sampling=config.evaluation.negative_sampling,
    neg_alpha=config.evaluation.neg_alpha,
)
chosen is sampled

True

### Contextual

A context-aware model is scored per test *row*, in its context, so its loaders
iterate over the rows of the evaluation split instead of over users. The full one
yields `(user, item, context)`; the sampled one adds `num_negatives` candidates
per row, drawn outside the user's training items and the row's own item. The
`contextual` dataset was built with the constructor's default `batch_size`,
1024, which is what its batches have.

In [32]:
batch = next(iter(contextual.get_contextual_evaluation_dataloader()))
show(batch, ["users", "items", "contexts"])

,shape,dtype,first row
users,"(1024,)",int64,[0]
items,"(1024,)",int64,[4]
contexts,"(1024, 2)",float32,"[1.0, 1.0]"


In [33]:
batch = next(
    iter(contextual.get_sampled_contextual_evaluation_dataloader(num_negatives=99))
)
show(batch, ["users", "test item", "candidates", "contexts"])

,shape,dtype,first row
users,"(1024,)",int64,[0]
test item,"(1024, 1)",int64,[4]
candidates,"(1024, 99)",int64,"[1126, 1459, 860, 1294, 1130, 1095, 1044, 1638, …]"
contexts,"(1024, 2)",float32,"[1.0, 1.0]"


The contextual loaders are chosen when the model is context-aware and the
dataset has contexts; any other model on the same dataset gets the per-user
loaders above.

## Summary

| key | what it changes in the loaders |
|---|---|
| `training.negative_sampling` | `uniform` or `popularity` draw of training negatives (interaction and transaction loaders; the session loaders always draw uniformly) |
| `training.neg_alpha` | the exponent on the training counts under `popularity`; 0 is uniform |
| `models.<Model>.batch_size` | the batch size of that model's training loader |
| `models.<Model>.neg_samples`, `max_seq_len`, `mask_prob` | negatives per sample, history length and masking share, for the models that have them |
| `evaluation.batch_size` | the batch size of every evaluation loader |
| `evaluation.strategy` | `full` or `sampled` evaluation loader |
| `evaluation.num_negatives` | candidates per user (per row, for contextual models) under `sampled` |
| `evaluation.negative_sampling`, `evaluation.neg_alpha` | how those candidates are drawn |

The keys are described on the [Training](https://warprec.readthedocs.io/en/latest/configuration/training/),
[Evaluation](https://warprec.readthedocs.io/en/latest/configuration/evaluation/)
and [Models](https://warprec.readthedocs.io/en/latest/configuration/models/)
configuration pages; the loaders themselves on the
[Entities API reference](https://warprec.readthedocs.io/en/latest/api-reference/data/entities/).
Guide 9 trains one model of each family on these loaders, and guide 12 covers
the evaluator that consumes the evaluation loaders.